# 02 - Suavizamiento Exponencial y ARIMA / SARIMA

**Módulo 3 - Series de Tiempo | ML Avanzado**

Los dos pilares del pronóstico estadístico clásico:

1. **Suavizamiento exponencial** (SES → Holt → Holt-Winters): promedios
   ponderados del pasado con pesos que decaen exponencialmente; descomponen la
   serie en nivel / tendencia / estacionalidad y los actualizan de forma
   recursiva.
2. **ARIMA / SARIMA**: modelan la serie (diferenciada hasta ser estacionaria)
   como función lineal de sus valores pasados y de sus errores pasados.

Ambas familias se evalúan con las métricas del módulo sobre el **mismo
horizonte de 60 días**, y todos los modelos quedan **registrados en MLflow**
(tracking + Model Registry).

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")

# Hacemos importable utils/ tanto si el notebook corre desde notebooks/ como
# desde la raíz del repositorio.
_here = os.getcwd()
for cand in (os.path.join(_here, "..", "utils"), os.path.join(_here, "utils"),
             os.path.join(_here, "..", "..", "module3-time-series", "utils")):
    cand = os.path.abspath(cand)
    if os.path.isdir(cand) and cand not in sys.path:
        sys.path.insert(0, cand)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from mlflow_helpers import setup_mlflow, log_and_register, register_best_run

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True
np.random.seed(42)
print("Versión de MLflow:", mlflow.__version__)

In [ ]:
# ---------------------------------------------------------------------------
# Carga del dataset UCI #235 (con caché local y respaldo sintético offline)
# ---------------------------------------------------------------------------
import io, zipfile, urllib.request

UCI_ZIP_URL = ("https://archive.ics.uci.edu/static/public/235/"
               "individual+household+electric+power+consumption.zip")

def _data_dir():
    for cand in ("../data", "data", "module3-time-series/data"):
        cand = os.path.abspath(cand)
        if os.path.isdir(cand):
            return cand
    cand = os.path.abspath("../data")
    os.makedirs(cand, exist_ok=True)
    return cand

DATA_DIR = _data_dir()
DAILY_CSV = os.path.join(DATA_DIR, "household_power_daily.csv")
HOURLY_CSV = os.path.join(DATA_DIR, "household_power_hourly.csv")

def load_household_power():
    """Devuelve (daily, hourly): potencia activa global media, en kW."""
    if os.path.isfile(DAILY_CSV) and os.path.isfile(HOURLY_CSV):
        daily = pd.read_csv(DAILY_CSV, index_col=0, parse_dates=True).iloc[:, 0]
        hourly = pd.read_csv(HOURLY_CSV, index_col=0, parse_dates=True).iloc[:, 0]
        return daily.asfreq("D"), hourly.asfreq("h")

    print("Descargando el dataset UCI #235 (~20 MB)...")
    raw = urllib.request.urlopen(UCI_ZIP_URL, timeout=180).read()
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        with zf.open("household_power_consumption.txt") as fh:
            df = pd.read_csv(fh, sep=";", na_values=["?"], low_memory=False,
                             usecols=["Date", "Time", "Global_active_power"])
    ts = pd.to_datetime(df["Date"] + " " + df["Time"],
                        format="%d/%m/%Y %H:%M:%S")
    power = pd.Series(df["Global_active_power"].astype(float).to_numpy(),
                      index=ts, name="global_active_power_kw").sort_index()

    # Agregamos y rellenamos huecos por interpolación temporal (~1.25% de
    # minutos faltantes + un par de cortes de varios días).
    daily = power.resample("D").mean().interpolate(method="time")
    hourly = power.resample("h").mean().interpolate(method="time")
    daily = daily.iloc[1:-1]                       # primer/último día parciales
    hourly = hourly.loc[daily.index.min():
                        daily.index.max() + pd.Timedelta(hours=23)]
    daily.to_frame().to_csv(DAILY_CSV)
    hourly.to_frame().to_csv(HOURLY_CSV)
    return daily.asfreq("D"), hourly.asfreq("h")

try:
    daily, hourly = load_household_power()
    print(f"daily : {daily.index.min().date()} .. {daily.index.max().date()} "
          f"(n={len(daily)})")
    print(f"hourly: n={len(hourly)}")
except Exception as e:
    print("No se pudo descargar el dataset:", repr(e))
    print("Usando RESPALDO SINTÉTICO (estacionalidad semanal + anual).")
    rng = np.random.default_rng(7)
    idx = pd.date_range("2006-12-17", "2010-11-25", freq="D")
    t = np.arange(len(idx))
    daily = pd.Series(
        1.1
        + 0.35 * np.cos(2 * np.pi * (t - 20) / 365.25)   # invierno alto
        + 0.10 * (idx.dayofweek >= 5)                     # fin de semana
        + rng.normal(0, 0.12, len(idx)),
        index=idx, name="global_active_power_kw").clip(lower=0.1).asfreq("D")
    hidx = pd.date_range(idx.min(), idx.max() + pd.Timedelta(hours=23), freq="h")
    hh = hidx.hour.to_numpy()
    base = daily.reindex(pd.DatetimeIndex(hidx.date)).to_numpy()
    profile = 0.6 + 0.35 * np.sin(2 * np.pi * (hh - 14) / 24) \
              + 0.25 * ((hh >= 18) & (hh <= 22))
    hourly = pd.Series(base * profile + rng.normal(0, 0.05, len(hidx)),
                       index=hidx, name=daily.name).clip(lower=0.05).asfreq("h")

In [ ]:
# ---------------------------------------------------------------------------
# Métricas de pronóstico + gráfico estándar — se usan en TODOS los notebooks.
# ---------------------------------------------------------------------------
def forecast_metrics(y_true, y_pred):
    """MSE, RMSE, MAE, MAPE y sMAPE como dict {nombre: float}."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    err = y_true - y_pred
    mse = float(np.mean(err ** 2))
    return {
        "MSE":   mse,
        "RMSE":  float(np.sqrt(mse)),
        "MAE":   float(np.mean(np.abs(err))),
        "MAPE":  float(np.mean(np.abs(err) / np.abs(y_true)) * 100.0),
        "sMAPE": float(np.mean(2.0 * np.abs(err)
                               / (np.abs(y_true) + np.abs(y_pred))) * 100.0),
    }

def print_metrics(name, m):
    print(f"{name:<26s} MSE={m['MSE']:.4f}  RMSE={m['RMSE']:.4f}  "
          f"MAE={m['MAE']:.4f}  MAPE={m['MAPE']:.2f}%  sMAPE={m['sMAPE']:.2f}%")

def metrics_table(metrics_by_model):
    """dict {modelo: dict_de_métricas} -> DataFrame ordenado por sMAPE."""
    return (pd.DataFrame(metrics_by_model).T
            .sort_values("sMAPE").round(4))

def plot_forecast(train, test, forecasts, title="", tail=180, ci=None):
    """Cola del train + test real + uno o varios pronósticos.

    forecasts : dict {nombre: pd.Series indexada como test}
    ci        : tupla opcional (lower, upper) para una banda de confianza
    Devuelve la figura (útil para loggearla en MLflow).
    """
    fig, ax = plt.subplots(figsize=(13, 5))
    train.iloc[-tail:].plot(ax=ax, label="train (cola)", color="0.65")
    test.plot(ax=ax, label="real (test)", color="black", lw=2)
    for name, fc in forecasts.items():
        fc.plot(ax=ax, label=name, lw=1.8)
    if ci is not None:
        ax.fill_between(test.index, ci[0], ci[1], alpha=0.2, label="IC 95%")
    ax.set_title(title)
    ax.set_ylabel("potencia activa media (kW)")
    ax.legend()
    plt.tight_layout()
    plt.show()
    return fig

In [ ]:
H = 60
train, test = daily.iloc[:-H], daily.iloc[-H:]
print(f"train: {train.index.min().date()} .. {train.index.max().date()} (n={len(train)})")
print(f"test : {test.index.min().date()} .. {test.index.max().date()} (n={len(test)})")

forecasts, all_metrics, fitted = {}, {}, {}   # acumularemos todos los modelos aquí

## 1. Suavizamiento Exponencial Simple (SES)

Idea: el pronóstico es un **promedio ponderado de todo el pasado**, con pesos
que decaen exponencialmente — lo reciente pesa más. Con nivel $\ell_t$ y
parámetro de suavizamiento $\alpha \in (0, 1]$:

$$
\ell_t = \alpha\, y_t + (1-\alpha)\, \ell_{t-1},
\qquad
\hat y_{T+h|T} = \ell_T \quad \forall h .
$$

Desenrollando la recursión: $\ell_t = \alpha \sum_{j=0}^{\infty} (1-\alpha)^j
y_{t-j}$ — de ahí el nombre *exponencial*.

- $\alpha \to 1$: solo importa el último valor (≈ naive).
- $\alpha \to 0$: promedio de largo plazo, muy suave.
- El pronóstico es **plano**: SES no modela tendencia ni estacionalidad.

`statsmodels` estima $\alpha$ (y el nivel inicial) maximizando la
verosimilitud.

In [ ]:
from statsmodels.tsa.holtwinters import (
    SimpleExpSmoothing, Holt, ExponentialSmoothing)

ses = SimpleExpSmoothing(train, initialization_method="estimated").fit()
fc = ses.forecast(H); fc.index = test.index
forecasts["SES"], fitted["SES"] = fc, ses
all_metrics["SES"] = forecast_metrics(test, fc)

print("alpha óptimo:", round(float(ses.params["smoothing_level"]), 4))
print_metrics("SES", all_metrics["SES"])

## 2. Holt: agregando tendencia (y amortiguándola)

Holt añade una componente de **tendencia** $b_t$ con su propio suavizamiento
$\beta$:

$$
\ell_t = \alpha\, y_t + (1-\alpha)(\ell_{t-1} + b_{t-1})
\qquad
b_t = \beta\, (\ell_t - \ell_{t-1}) + (1-\beta)\, b_{t-1}
$$

$$
\hat y_{T+h|T} = \ell_T + h\, b_T .
$$

Extrapolar una recta indefinidamente suele ser demasiado optimista; la
variante **amortiguada** (*damped*, Gardner) multiplica la tendencia por
$\phi \in (0,1)$: $\hat y_{T+h|T} = \ell_T + (\phi + \phi^2 + \dots + \phi^h)
b_T$, que converge a una asíntota. En la práctica el *damped trend* es uno de
los pronosticadores univariados más difíciles de vencer.

In [ ]:
holt = Holt(train, damped_trend=True, initialization_method="estimated").fit()
fc = holt.forecast(H); fc.index = test.index
forecasts["Holt"], fitted["Holt"] = fc, holt
all_metrics["Holt"] = forecast_metrics(test, fc)

print({k: round(float(v), 4) for k, v in holt.params.items()
       if k in ("smoothing_level", "smoothing_trend", "damping_trend")})
print_metrics("Holt (damped)", all_metrics["Holt"])

## 3. Holt-Winters: agregando estacionalidad

La versión completa añade un componente **estacional** $s_t$ de periodo $m$
(aquí $m=7$, la semana) con suavizamiento $\gamma$. En la forma **aditiva**:

$$
\ell_t = \alpha\,(y_t - s_{t-m}) + (1-\alpha)(\ell_{t-1} + b_{t-1})
$$
$$
b_t = \beta\,(\ell_t - \ell_{t-1}) + (1-\beta)\, b_{t-1}
\qquad
s_t = \gamma\,(y_t - \ell_{t-1} - b_{t-1}) + (1-\gamma)\, s_{t-m}
$$
$$
\hat y_{T+h|T} = \ell_T + h\, b_T + s_{T+h-m\lceil h/m \rceil} .
$$

La forma **multiplicativa** reemplaza restas por divisiones
($y_t / s_{t-m}$, etc.) y se usa cuando la amplitud estacional escala con el
nivel. Nuestra serie es aproximadamente aditiva (notebook 01), así que usamos
`seasonal="add"` con tendencia amortiguada.

In [ ]:
hw = ExponentialSmoothing(
    train, trend="add", damped_trend=True,
    seasonal="add", seasonal_periods=7,
    initialization_method="estimated").fit()
fc = hw.forecast(H); fc.index = test.index
forecasts["Holt-Winters"], fitted["Holt-Winters"] = fc, hw
all_metrics["Holt-Winters"] = forecast_metrics(test, fc)

print({k: round(float(v), 4) for k, v in hw.params.items()
       if k in ("smoothing_level", "smoothing_trend",
                "smoothing_seasonal", "damping_trend")})
print_metrics("Holt-Winters", all_metrics["Holt-Winters"])

In [ ]:
fig_es = plot_forecast(
    train, test,
    {k: forecasts[k] for k in ("SES", "Holt", "Holt-Winters")},
    title="Familia de suavizamiento exponencial (test = últimos 60 días)")
# SES y Holt son (casi) planos; Holt-Winters recupera el patrón semanal.

## 4. ARIMA: los bloques de construcción

Recordatorio del operador de rezago: $L\, y_t = y_{t-1}$, $L^k y_t = y_{t-k}$.

**AR(p)** — el presente como combinación lineal del pasado reciente:
$$
y_t = c + \phi_1 y_{t-1} + \dots + \phi_p y_{t-p} + \varepsilon_t
\quad\Longleftrightarrow\quad
\phi(L)\, y_t = c + \varepsilon_t .
$$

**MA(q)** — el presente como combinación de los *shocks* recientes:
$$
y_t = c + \varepsilon_t + \theta_1 \varepsilon_{t-1} + \dots + \theta_q \varepsilon_{t-q}
= c + \theta(L)\, \varepsilon_t .
$$

**ARMA(p, q)** — ambos: $\phi(L)\, y_t = c + \theta(L)\, \varepsilon_t$.
Supone **estacionariedad**.

**ARIMA(p, d, q)** — la "I" de *integrada*: diferenciamos $d$ veces primero:
$$
\phi(L)\,(1-L)^d\, y_t = c + \theta(L)\, \varepsilon_t .
$$

**SARIMA$(p,d,q)(P,D,Q)_m$** — polinomios estacionales en $L^m$ y diferencia
estacional $(1-L^m)^D$:
$$
\phi(L)\,\Phi(L^m)\,(1-L)^d (1-L^m)^D\, y_t
= c + \theta(L)\,\Theta(L^m)\,\varepsilon_t .
$$

### Identificación con ACF/PACF (serie ya estacionaria)

| Patrón | Sugiere |
|---|---|
| ACF se corta tras el rezago $q$; PACF decae | **MA(q)** |
| PACF se corta tras el rezago $p$; ACF decae | **AR(p)** |
| Ambas decaen gradualmente | **ARMA(p, q)** |
| ACF decae *muy* lento | falta diferenciar ($d$+1) |
| Picos en los rezagos $m, 2m, \dots$ | términos estacionales $P/Q$ |

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

# ¿Cuánta diferenciación necesitamos?
print(f"ADF serie original      : p = {adfuller(train, autolag='AIC')[1]:.4f}")
diffed = train.diff(7).dropna()           # d=0, D=1 (m=7)
print(f"ADF tras (1-L^7)        : p = {adfuller(diffed, autolag='AIC')[1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
plot_acf(diffed, lags=42, ax=axes[0]); axes[0].set_title("ACF de (1-L^7) y_t")
plot_pacf(diffed, lags=42, ax=axes[1], method="ywm"); axes[1].set_title("PACF")
plt.tight_layout(); plt.show()

# Lectura: el ADF de la serie original ya rechaza raíz unitaria (no hay
# tendencia de largo plazo) -> d=0. La estacionalidad semanal pide D=1 con
# m=7. En la serie diferenciada: pico negativo en el rezago 7 de la ACF
# (MA estacional Q=1) y autocorrelación de corto plazo (p=1, q=1).
# Candidato: SARIMA(1,0,1)(0,1,1)_7.

## 5. Ajuste del SARIMA y diagnóstico de residuos

Ajustamos SARIMA$(1,0,1)(0,1,1)_7$. Si el modelo capturó la estructura, los
residuos deben ser **ruido blanco**:

- **Ljung-Box**: $H_0$ = residuos independientes hasta el rezago $h$;
  *queremos* p-valores **altos** (no rechazar).
- **Q-Q plot**: puntos sobre la recta → residuos ~normales.
- **ACF de residuos**: dentro de la banda de confianza.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

ORDER, SORDER = (1, 0, 1), (0, 1, 1, 7)
sarima = SARIMAX(train, order=ORDER, seasonal_order=SORDER,
                 enforce_stationarity=False, enforce_invertibility=False
                 ).fit(disp=False)
print(sarima.summary().tables[1])

In [ ]:
fig = sarima.plot_diagnostics(figsize=(12, 8))
plt.tight_layout(); plt.show()

from statsmodels.stats.diagnostic import acorr_ljungbox
print(acorr_ljungbox(sarima.resid, lags=[7, 14, 28], return_df=True))
print("\nQueremos lb_pvalue GRANDES -> residuos ~ ruido blanco.")

## 6. Pronóstico SARIMA con intervalos de confianza

`get_forecast` devuelve la media y un **intervalo de confianza** por paso; el
intervalo se ensancha con el horizonte porque la incertidumbre se acumula.

In [ ]:
fc_obj = sarima.get_forecast(steps=H)
fc = fc_obj.predicted_mean; fc.index = test.index
ci = fc_obj.conf_int(alpha=0.05); ci.index = test.index

forecasts["SARIMA"], fitted["SARIMA"] = fc, sarima
all_metrics["SARIMA"] = forecast_metrics(test, fc)
print_metrics("SARIMA(1,0,1)(0,1,1)7", all_metrics["SARIMA"])

fig_sarima = plot_forecast(
    train, test, {"SARIMA": fc},
    title="SARIMA(1,0,1)(0,1,1)$_7$ con IC 95%",
    ci=(ci.iloc[:, 0], ci.iloc[:, 1]))

## 7. `auto_arima` (pmdarima) — selección automática (opcional)

`pmdarima.auto_arima` busca los órdenes automáticamente: $d$/$D$ con pruebas
de raíz unitaria (ADF/KPSS, OCSB/CH) y $p,q,P,Q$ minimizando AIC/BIC con una
búsqueda *stepwise*. Trátalo con sentido crítico: corre siempre los mismos
diagnósticos de residuos. (Celda opcional — `pip install pmdarima`.)

In [ ]:
try:
    import pmdarima as pm
    auto = pm.auto_arima(train, seasonal=True, m=7, stepwise=True,
                         suppress_warnings=True, error_action="ignore",
                         max_p=3, max_q=3, max_P=2, max_Q=2)
    print("auto_arima eligió:", auto.order, auto.seasonal_order,
          "| AIC:", round(auto.aic(), 1))
except Exception as e:
    print("pmdarima no disponible (opcional):", repr(e))
    print("Seguimos con el SARIMA(1,0,1)(0,1,1)_7 identificado a mano.")

## 8. Comparación de todos los modelos

In [ ]:
table = metrics_table(all_metrics)
display(table)

ax = table["sMAPE"].plot(kind="barh", figsize=(8, 3.5), color="C0")
ax.set_xlabel("sMAPE (%)  (menor = mejor)")
ax.set_title("Suavizamiento exponencial vs SARIMA - test de 60 días")
plt.tight_layout(); plt.show()

fig_all = plot_forecast(train, test, forecasts,
                        title="Todos los modelos estadísticos vs realidad")

## 9. Tracking y Registry en MLflow

Un **run por modelo** con: hiperparámetros, las 5 métricas, la figura del
pronóstico comparativo y el propio modelo serializado (flavor
`mlflow.statsmodels`). Al final promovemos el mejor (mínimo sMAPE) al **Model
Registry** con `register_best_run` — igual que en el Módulo 2.

In [ ]:
setup_mlflow("module3-02-statistical-models", backend="dagshub")

def es_params(fit_result):
    keys = ("smoothing_level", "smoothing_trend",
            "smoothing_seasonal", "damping_trend")
    out = {}
    for k in keys:
        v = fit_result.params.get(k, np.nan)
        if v == v:                       # descarta NaN
            out[k] = round(float(v), 4)
    return out

run_params = {
    "SES":          {"model": "SimpleExpSmoothing", **es_params(fitted["SES"])},
    "Holt":         {"model": "Holt", "damped": True, **es_params(fitted["Holt"])},
    "Holt-Winters": {"model": "ExponentialSmoothing", "trend": "add",
                     "damped": True, "seasonal": "add", "m": 7,
                     **es_params(fitted["Holt-Winters"])},
    "SARIMA":       {"model": "SARIMAX", "order": str(ORDER),
                     "seasonal_order": str(SORDER)},
}

for name in forecasts:
    log_and_register(
        run_name=name,
        params={**run_params[name], "horizon_days": H,
                "dataset": "uci-household-power"},
        metrics=all_metrics[name],
        model=fitted[name],
        flavor="statsmodels",
        tags={"notebook": "02_arima", "familia": "estadistica"},
        figures={"plots/comparacion.png": fig_all},
    )

register_best_run("module3-02-statistical-models", metric="sMAPE",
                  registered_model_name="module3-power-statistical",
                  mode="min")

## 10. Serving: consumir el modelo desde el Registry

Cerramos el **ciclo de gestión del modelo**: entrenar → trackear → registrar
→ **servir**. Un proceso consumidor (una API, un job batch de pronóstico) no
reentrena nada ni conoce este notebook: solo necesita el **nombre** del modelo
en el registry y pide la última versión con la URI

```
models:/module3-power-statistical/latest
```

Usamos el flavor **nativo** (`mlflow.statsmodels.load_model`) porque devuelve
el objeto de resultados de statsmodels con su API completa
(`.forecast(steps)`, intervalos de confianza...); el wrapper genérico
`pyfunc` espera un DataFrame de entrada y no encaja con la firma
`predict(start, end)` de los modelos estadísticos.

In [ ]:
MODEL_NAME = "module3-power-statistical"
MODEL_URI = f"models:/{MODEL_NAME}/latest"

client = mlflow.MlflowClient()
versions = client.search_model_versions(f"name = '{MODEL_NAME}'")
latest = max(int(v.version) for v in versions)
print(f"Registry: '{MODEL_NAME}' tiene {len(versions)} versión(es); "
      f"sirviendo la v{latest}")

serving_model = mlflow.statsmodels.load_model(MODEL_URI)
print("Modelo cargado:", type(serving_model.model).__name__)

# El modelo registrado se ajustó sobre `train`: su forecast arranca justo
# donde termina el entrenamiento, es decir, sobre la ventana de test.
fc_serving = serving_model.forecast(steps=H)
fc_serving.index = test.index

m_serving = forecast_metrics(test, fc_serving)
print_metrics("modelo servido (registry)", m_serving)

plot_forecast(train, test, {"pronóstico servido": fc_serving},
              title=f"Serving desde el registry: {MODEL_NAME} v{latest}");

En producción el patrón es el mismo pero con dos diferencias:

1. **Reajuste con datos frescos**: antes de pronosticar el futuro real, el
   modelo se reentrena (o se actualiza con `.append()` en statsmodels) con
   todas las observaciones disponibles — aquí mantuvimos el ajuste original
   para poder comparar contra el test.
2. **Versiones explícitas**: en vez de `latest`, un servicio serio fija la
   versión (`models:/nombre/3`) o usa un *alias* (`@champion`) que se mueve
   solo tras validar la nueva versión.

## Resumen

- **SES** suaviza solo el nivel (pronóstico plano); **Holt** añade tendencia
  (mejor amortiguada con $\phi$); **Holt-Winters** añade estacionalidad
  aditiva o multiplicativa — aquí $m=7$.
- **ARIMA**: AR usa valores pasados, MA usa shocks pasados, la "I" diferencia
  hasta la estacionariedad; **SARIMA** añade $(P,D,Q)_m$. Órdenes: $d, D$ por
  pruebas ADF + inspección; $p, q, P, Q$ por ACF/PACF (o `auto_arima`).
- Valida SIEMPRE los residuos (Ljung-Box, Q-Q, ACF) y reporta **intervalos de
  confianza**.
- Todos los modelos quedaron en **MLflow** y el mejor (por sMAPE) en el
  **Model Registry** — y cerramos el ciclo **consumiéndolo** desde
  `models:/module3-power-statistical/latest` para pronosticar.

Siguiente: **03 — Ingeniería de variables para series de tiempo**, el puente
hacia el pronóstico con machine learning.